In [3]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
import matplotlib.pyplot as plt
import pandas as pd
import gc

In [4]:
# ==================== CLEAR MEMORY ====================
tf.keras.backend.clear_session()
gc.collect()

0

In [5]:
# ==================== PARAMETERS ====================
SAVE_PATH = "windowed_data"
WINDOW_SIZE = 24
HORIZON = 1
EPOCHS = 15
NUM_BATCHES = 5
RANDOM_STATE = 42

In [6]:
# ==================== 1. LOAD DATA ====================
X_all = []
y_all = []

for b in range(NUM_BATCHES):
    X_batch = np.load(os.path.join(SAVE_PATH, f"X_batch_{b}.npy"))
    y_batch = np.load(os.path.join(SAVE_PATH, f"y_batch_{b}.npy"))
    X_all.append(X_batch)
    y_all.append(y_batch)

X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

print(f"Total samples loaded: {len(X_all)}")
print(f"Shape: X={X_all.shape}, y={y_all.shape}")

Total samples loaded: 164424
Shape: X=(164424, 24), y=(164424, 1)


In [7]:
# ==================== 2. SPLIT DATA: 70% TRAIN, 15% VAL, 15% TEST ====================
n_samples = len(X_all)
train_size = int(0.70 * n_samples)
val_size = int(0.15 * n_samples)

X_train_flat = X_all[:train_size]
y_train = y_all[:train_size].flatten()  # FLATTEN HERE

X_val_flat = X_all[train_size:train_size + val_size]
y_val = y_all[train_size:train_size + val_size].flatten()  # FLATTEN HERE

X_test_flat = X_all[train_size + val_size:]
y_test = y_all[train_size + val_size:].flatten()  # FLATTEN HERE

print(f"Train: {len(X_train_flat)} samples ({len(X_train_flat)/n_samples*100:.1f}%)")
print(f"Val:   {len(X_val_flat)} samples ({len(X_val_flat)/n_samples*100:.1f}%)")
print(f"Test:  {len(X_test_flat)} samples ({len(X_test_flat)/n_samples*100:.1f}%)")

del X_all, y_all
gc.collect()

Train: 115096 samples (70.0%)
Val:   24663 samples (15.0%)
Test:  24665 samples (15.0%)


0

In [8]:
# ==================== 3. FEATURE ENGINEERING FUNCTIONS ====================
def create_manual_features(X):
    """Create hand-crafted statistical features"""
    features = []
    features.append(np.mean(X, axis=1))
    features.append(np.std(X, axis=1))
    features.append(np.min(X, axis=1))
    features.append(np.max(X, axis=1))
    features.append(X[:, -1])
    features.append(X[:, -1] - X[:, 0])
    features.append(np.median(X, axis=1))
    features.append(np.percentile(X, 25, axis=1))
    features.append(np.percentile(X, 75, axis=1))
    return np.column_stack(features)

def apply_fisher_score(X_train, y_train, X_val, X_test, k=10):
    """Fisher Score (ANOVA F-test) feature selection"""
    selector = SelectKBest(score_func=f_regression, k=k)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_val_selected = selector.transform(X_val)
    X_test_selected = selector.transform(X_test)
    return X_train_selected, X_val_selected, X_test_selected

def apply_mrmr(X_train, y_train, X_val, X_test, k=10):
    """mRMR approximation using Mutual Information"""
    selector = SelectKBest(score_func=mutual_info_regression, k=k)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_val_selected = selector.transform(X_val)
    X_test_selected = selector.transform(X_test)
    return X_train_selected, X_val_selected, X_test_selected

def apply_pca(X_train, X_val, X_test, n_components=10):
    """PCA dimensionality reduction"""
    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train)
    X_val_pca = pca.transform(X_val)
    X_test_pca = pca.transform(X_test)
    print(f"  PCA variance explained: {pca.explained_variance_ratio_.sum()*100:.2f}%")
    return X_train_pca, X_val_pca, X_test_pca



In [ ]:
# ==================== 4. BUILD & TRAIN GRU MODEL ====================
def build_and_train_GRU(X_train, y_train, X_val, y_val, model_name, input_dim):
    """Build and train GRU model with memory efficiency"""
    print(f"\n  Training {model_name}...")

    # Clear previous model from memory
    tf.keras.backend.clear_session()
    gc.collect()

    # Reshape for GRU if needed
    if X_train.ndim == 2:
        X_train = X_train[..., np.newaxis]
        X_val = X_val[..., np.newaxis]

    model = Sequential(
        [
            GRU(64, input_shape=(X_train.shape[1], 1)),
            Dense(1),
        ]
    )

    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")

    early_stop = EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=0
    )

    # Train with SMALL batch size for memory
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=64,  # Small batch size to avoid freeze
        callbacks=[early_stop],
        verbose=1,  # Show progress
        shuffle=True,
    )

    print(f"  ✓ {model_name} trained (epochs: {len(history.history['loss'])})")
    gc.collect()

    return model, history

In [16]:
# ==================== 5. EVALUATION FUNCTION ====================
def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate model in batches to save memory"""
    if X_test.ndim == 2:
        X_test = X_test[..., np.newaxis]
    
    # Predict in small batches
    y_pred = model.predict(X_test, verbose=0, batch_size=64).reshape(-1)
    
    errors = y_test - y_pred
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    
    delta = 15
    accuracy = (np.sum(np.abs(errors) <= delta) / len(errors)) * 100
    
    # MAPE calculation
    mape = np.mean(np.abs(errors) / np.abs(y_test)) * 100
    mape_accuracy = 100 - mape

    
    return {
        'Model': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'Accuracy (%)': accuracy,
        'MAPE (%)': mape,
        'MAPE Accuracy (%)': mape_accuracy
    }

In [17]:
# ==================== 6. BASELINE MODEL (NAIVE PREDICTION) ====================
y_naive = X_test_flat[:, -1]
errors_naive = y_test - y_naive
mse_naive = np.mean(errors_naive ** 2)
rmse_naive = np.sqrt(mse_naive)
mae_naive = np.mean(np.abs(errors_naive))

delta = 15
correct_count = np.sum(np.abs(errors_naive) <= delta)
total_count = len(errors_naive)
acc_naive = (correct_count / total_count) * 100

# MAPE
mape_naive = np.mean(np.abs(errors_naive) / np.abs(y_test)) * 100
mape_acc_naive = 100 - mape_naive

baseline_results = {
    'Model': 'Baseline (Naive)',
    'MSE': mse_naive,
    'RMSE': rmse_naive,
    'MAE': mae_naive,
    'Accuracy (%)': acc_naive,
    'MAPE (%)': mape_naive,
    'MAPE Accuracy (%)': mape_acc_naive
}

print(f"Baseline RMSE: {rmse_naive:.2f} mg/dL")
print(f"Baseline MAE: {mae_naive:.2f} mg/dL")
print(f"Baseline Accuracy: {acc_naive:.2f}%")
print(f"Baseline MAPE: {mape_naive:.2f}%")
print(f"Baseline MAPE Accuracy: {mape_acc_naive:.2f}%")

del y_naive, errors_naive
gc.collect()

Baseline RMSE: 14.75 mg/dL
Baseline MAE: 9.72 mg/dL
Baseline Accuracy: 80.90%
Baseline MAPE: 6.41%
Baseline MAPE Accuracy: 93.59%


3214

In [18]:
# ==================== 7. MODEL 1: RAW FEATURES (NO FEATURE SELECTION) ====================
X_train_raw = X_train_flat.copy()
X_val_raw = X_val_flat.copy()
X_test_raw = X_test_flat.copy()

model_raw, hist_raw = build_and_train_GRU(X_train_raw, y_train, X_val_raw, y_val, 
                                            "Raw Features", X_train_raw.shape[1])
results_raw = evaluate_model(model_raw, X_test_raw, y_test, "Raw Features")

print(f"Raw RMSE: {results_raw['RMSE']:.2f} mg/dL")
print(f"Raw MAE: {results_raw['MAE']:.2f} mg/dL")
print(f"Raw Accuracy: {results_raw['Accuracy (%)']:.2f}%")
print(f"Raw MAPE: {results_raw['MAPE (%)']:.2f}%")
print(f"Raw MAPE Accuracy: {results_raw['MAPE Accuracy (%)']:.2f}%")

del X_train_raw, X_val_raw, X_test_raw, model_raw
gc.collect()


  Training Raw Features...
Epoch 1/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 43s 22ms/step - loss: 16151.3340 - val_loss: 5298.3726
Epoch 2/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - loss: 4452.4536 - val_loss: 1301.2408
Epoch 3/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - loss: 1495.2876 - val_loss: 455.0144
Epoch 4/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - loss: 596.1792 - val_loss: 225.9418
Epoch 5/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - loss: 300.2498 - val_loss: 166.1970
Epoch 6/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - loss: 185.7386 - val_loss: 127.1603
Epoch 7/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 40s 22ms/step - loss: 145.5378 - val_loss: 111.9533
Epoch 8/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 39s 22ms/step - loss: 131.6037 - val_loss: 115.6577
Epoch 9/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - loss: 126.6881 - val_loss: 107.8743
Epoch 10/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 28s 15ms/step - loss: 152.6070 - val_loss: 108.6256
Epoch 11/15
1799/1799 ━━━

331

In [19]:
# ==================== 8. MODEL 2: MANUAL FEATURES ====================
X_train_manual = create_manual_features(X_train_flat)
X_val_manual = create_manual_features(X_val_flat)
X_test_manual = create_manual_features(X_test_flat)


model_manual, hist_manual = build_and_train_GRU(X_train_manual, y_train, X_val_manual, y_val,
                                                  "Manual Features", X_train_manual.shape[1])
results_manual = evaluate_model(model_manual, X_test_manual, y_test, "Manual Features")

print(f"Manual RMSE: {results_manual['RMSE']:.2f} mg/dL")
print(f"Manual MAE: {results_manual['MAE']:.2f} mg/dL")
print(f"Manual Accuracy: {results_manual['Accuracy (%)']:.2f}%")
print(f"Manual MAPE: {results_manual['MAPE (%)']:.2f}%")
print(f"Manual MAPE Accuracy: {results_manual['MAPE Accuracy (%)']:.2f}%")

del X_train_manual, X_val_manual, X_test_manual, model_manual
gc.collect()


  Training Manual Features...
Epoch 1/15


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 16350.3164 - val_loss: 5559.9849
Epoch 2/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 4773.5347 - val_loss: 1455.1000
Epoch 3/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 1660.1815 - val_loss: 648.1053
Epoch 4/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 756.8289 - val_loss: 390.8359
Epoch 5/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 476.6801 - val_loss: 355.5031
Epoch 6/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 393.3639 - val_loss: 325.2994
Epoch 7/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 363.9381 - val_loss: 326.4180
Epoch 8/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 355.0253 - val_loss: 323.8103
Epoch 9/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 350.0110 - val_loss: 311.0346
Epoch 10/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 345.1490 - val_loss: 310.6490
Epoch 11/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 344.9139 - val_loss:

331

In [20]:
# ==================== 9. MODEL 3: FISHER SCORE ====================
K_FISHER = 10
X_train_fisher, X_val_fisher, X_test_fisher = apply_fisher_score(
    X_train_flat, y_train, X_val_flat, X_test_flat, k=K_FISHER
)
print(f"  Fisher: Selected top {K_FISHER} features")

model_fisher, hist_fisher = build_and_train_GRU(X_train_fisher, y_train, X_val_fisher, y_val,
                                                  "Fisher Score", X_train_fisher.shape[1])
results_fisher = evaluate_model(model_fisher, X_test_fisher, y_test, "Fisher Score")

print(f"Fisher RMSE: {results_fisher['RMSE']:.2f} mg/dL")
print(f"Fisher MAE: {results_fisher['MAE']:.2f} mg/dL")
print(f"Fisher Accuracy: {results_fisher['Accuracy (%)']:.2f}%")
print(f"Fisher MAPE: {results_fisher['MAPE (%)']:.2f}%")
print(f"Fisher MAPE Accuracy: {results_fisher['MAPE Accuracy (%)']:.2f}%")

del X_train_fisher, X_val_fisher, X_test_fisher, model_fisher
gc.collect()

  Fisher: Selected top 10 features

  Training Fisher Score...
Epoch 1/15


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - loss: 16137.5410 - val_loss: 5310.0449
Epoch 2/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 4475.0293 - val_loss: 1317.0897
Epoch 3/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 1511.0043 - val_loss: 454.9906
Epoch 4/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 639.2823 - val_loss: 248.6611
Epoch 5/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 313.7447 - val_loss: 156.3346
Epoch 6/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 198.9300 - val_loss: 143.5541
Epoch 7/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 161.6773 - val_loss: 121.8661
Epoch 8/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 148.8284 - val_loss: 122.0385
Epoch 9/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 135.5820 - val_loss: 117.5068
Epoch 10/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 132.1943 - val_loss: 114.7736
Epoch 11/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 128.5535 - val_loss

331

In [21]:
# ==================== 10. MODEL 4: Mutual mRMR ====================
K_MRMR = 10
X_train_mrmr, X_val_mrmr, X_test_mrmr = apply_mrmr(
    X_train_flat, y_train, X_val_flat, X_test_flat, k=K_MRMR
)
print(f"  mRMR: Selected top {K_MRMR} features")

model_mrmr, hist_mrmr = build_and_train_GRU(X_train_mrmr, y_train, X_val_mrmr, y_val,
                                              "mRMR", X_train_mrmr.shape[1])
results_mrmr = evaluate_model(model_mrmr, X_test_mrmr, y_test, "mRMR")

print(f"mrmr RMSE: {results_mrmr['RMSE']:.2f} mg/dL")
print(f"mrmr MAE: {results_mrmr['MAE']:.2f} mg/dL")
print(f"mrmr Accuracy: {results_mrmr['Accuracy (%)']:.2f}%")
print(f"mrmr MAPE: {results_mrmr['MAPE (%)']:.2f}%")
print(f"mrmr MAPE Accuracy: {results_mrmr['MAPE Accuracy (%)']:.2f}%")


del X_train_mrmr, X_val_mrmr, X_test_mrmr, model_mrmr
gc.collect()

  mRMR: Selected top 10 features

  Training mRMR...
Epoch 1/15


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 15873.0156 - val_loss: 5156.5537
Epoch 2/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 4362.5127 - val_loss: 1269.0299
Epoch 3/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 1453.9775 - val_loss: 450.3120
Epoch 4/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 572.6036 - val_loss: 211.4914
Epoch 5/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 279.3698 - val_loss: 156.4755
Epoch 6/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 183.5124 - val_loss: 132.6428
Epoch 7/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 150.1307 - val_loss: 114.2928
Epoch 8/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 138.2423 - val_loss: 119.4036
Epoch 9/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 133.6505 - val_loss: 113.6315
Epoch 10/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 126.6920 - val_loss: 111.5787
Epoch 11/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 126.3831 - val_loss:

331

In [22]:
# ==================== 11. MODEL 5: PCA ====================
N_COMPONENTS = 10
X_train_pca, X_val_pca, X_test_pca = apply_pca(
    X_train_flat, X_val_flat, X_test_flat, n_components=N_COMPONENTS
)
print(f"  PCA: Reduced to {N_COMPONENTS} components")

model_pca, hist_pca = build_and_train_GRU(X_train_pca, y_train, X_val_pca, y_val,
                                            "PCA", X_train_pca.shape[1])
results_pca = evaluate_model(model_pca, X_test_pca, y_test, "PCA")

print(f"PCA RMSE: {results_pca['RMSE']:.2f} mg/dL")
print(f"PCA MAE: {results_pca['MAE']:.2f} mg/dL")
print(f"PCA Accuracy: {results_pca['Accuracy (%)']:.2f}%")
print(f"PCA MAPE: {results_pca['MAPE (%)']:.2f}%")
print(f"PCA MAPE Accuracy: {results_pca['MAPE Accuracy (%)']:.2f}%")

del X_train_pca, X_val_pca, X_test_pca, model_pca
gc.collect()

  PCA variance explained: 99.18%
  PCA: Reduced to 10 components

  Training PCA...
Epoch 1/15


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 15835.3291 - val_loss: 5367.1948
Epoch 2/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 5413.2524 - val_loss: 3030.8025
Epoch 3/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 3451.1411 - val_loss: 1568.1792
Epoch 4/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 1240.8250 - val_loss: 508.0919
Epoch 5/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 571.6057 - val_loss: 350.6381
Epoch 6/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 403.3933 - val_loss: 316.7587
Epoch 7/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 347.6560 - val_loss: 315.8389
Epoch 8/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 320.3958 - val_loss: 273.6033
Epoch 9/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 304.1043 - val_loss: 268.4941
Epoch 10/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 295.8546 - val_loss: 308.8858
Epoch 11/15
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 290.8094 - val_los

331

In [23]:
# ==================== 12. FINAL COMPARISON ====================
# Compile all results
all_results = [
    baseline_results,
    results_raw,
    results_manual,
    results_fisher,
    results_mrmr,
    results_pca
]

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('RMSE')

print("\n" + results_df.to_string(index=False))

# Find best model
best_model = results_df.iloc[0]['Model']
best_rmse = results_df.iloc[0]['RMSE']
best_acc = results_df.iloc[0]['Accuracy (%)']

print(f"BEST MODEL: {best_model}")
print(f"RMSE: {best_rmse:.2f} mg/dL")
print(f"Accuracy: {best_acc:.2f}%")


           Model        MSE      RMSE       MAE  Accuracy (%)  MAPE (%)  MAPE Accuracy (%)
    Raw Features 121.486801 11.022105  6.600804     90.780458  4.415787          95.584213
    Fisher Score 126.202736 11.233999  6.651248     90.139874  4.364470          95.635529
            mRMR 126.226006 11.235035  6.693953     90.139874  4.436152          95.563850
Baseline (Naive) 217.547867 14.749504  9.720819     80.904115  6.409017          93.590981
 Manual Features 239.369720 15.471578 10.751514     76.472735  7.379246          92.620758
             PCA 260.923584 16.153130 11.171751     74.486114  7.691268          92.308731
BEST MODEL: Raw Features
RMSE: 11.02 mg/dL
Accuracy: 90.78%


In [24]:
# ==================== 13. VISUALIZATIONS ====================
print("\n[10] GENERATING VISUALIZATIONS...")
print("-" * 70)

# Plot 1: Metrics Comparison (4 plots now)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# RMSE Bar Chart
axes[0, 0].bar(results_df['Model'], results_df['RMSE'], color='steelblue')
axes[0, 0].set_xlabel('Model', fontsize=12)
axes[0, 0].set_ylabel('RMSE (mg/dL)', fontsize=12)
axes[0, 0].set_title('RMSE Comparison', fontsize=14, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(axis='y', alpha=0.3)

# Accuracy Bar Chart
axes[0, 1].bar(results_df['Model'], results_df['Accuracy (%)'], color='coral')
axes[0, 1].set_xlabel('Model', fontsize=12)
axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
axes[0, 1].set_title('Accuracy Comparison (±15 mg/dL)', fontsize=14, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)

# MAE Bar Chart
axes[1, 0].bar(results_df['Model'], results_df['MAE'], color='seagreen')
axes[1, 0].set_xlabel('Model', fontsize=12)
axes[1, 0].set_ylabel('MAE (mg/dL)', fontsize=12)
axes[1, 0].set_title('MAE Comparison', fontsize=14, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(axis='y', alpha=0.3)

# MAPE Accuracy Bar Chart
axes[1, 1].bar(results_df['Model'], results_df['MAPE Accuracy (%)'], color='purple')
axes[1, 1].set_xlabel('Model', fontsize=12)
axes[1, 1].set_ylabel('MAPE Accuracy (%)', fontsize=12)
axes[1, 1].set_title('MAPE Accuracy (100 - MAPE)', fontsize=14, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: model_comparison.png")
plt.close()

# Plot 2: Training History
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
axes2 = axes2.flatten()

histories = [
    (hist_raw, 'Raw Features'),
    (hist_manual, 'Manual Features'),
    (hist_fisher, 'Fisher Score'),
    (hist_mrmr, 'mRMR'),
    (hist_pca, 'PCA')
]

for idx, (hist, name) in enumerate(histories):
    axes2[idx].plot(hist.history['loss'], label='Train Loss', linewidth=2)
    axes2[idx].plot(hist.history['val_loss'], label='Val Loss', linewidth=2)
    axes2[idx].set_xlabel('Epoch', fontsize=10)
    axes2[idx].set_ylabel('Loss (MSE)', fontsize=10)
    axes2[idx].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes2[idx].legend()
    axes2[idx].grid(True, alpha=0.3)

axes2[5].axis('off')

plt.tight_layout()
plt.savefig('training_histories.png', dpi=300, bbox_inches='tight')
print("✓ Saved: training_histories.png")
plt.close()


[10] GENERATING VISUALIZATIONS...
----------------------------------------------------------------------
✓ Saved: model_comparison.png
✓ Saved: training_histories.png


In [25]:
# ==================== 14. SAVE RESULTS ====================
results_df.to_csv('model_comparison_results.csv', index=False)
print("Saved: model_comparison_results.csv")

# Final cleanup
del X_train_flat, X_val_flat, X_test_flat, y_train, y_val, y_test
gc.collect()

Saved: model_comparison_results.csv


42